In [1]:
import pandas as pd
import numpy as np
import os

INPUT_DIR = "/kaggle/input/competitions/enveda-CASMI26-molecule-id-mass-spectra"

print("1. Cargando datos para el asalto con Anclaje de Pico Base...")
sub_df = pd.read_csv(os.path.join(INPUT_DIR, "sample_submission.csv"))
test_df = pd.read_parquet(os.path.join(INPUT_DIR, "test.parquet"))
train_df = pd.read_parquet(os.path.join(INPUT_DIR, "train.parquet"))

fallback_smiles = ";".join(train_df['normalized_smiles'].dropna().drop_duplicates().head(25).values)
db_by_adduct = {adduct: group for adduct, group in train_df.groupby('adduct')}

def clean_spectrum(mzs, ints, relative_threshold=0.009):
    if ints is None or len(ints) == 0:
        return np.array([]), np.array([])
    mzs = np.asarray(mzs)
    ints = np.asarray(ints)
    max_i = np.max(ints)
    if max_i == 0:
        return mzs, ints
    mask = (ints / max_i) >= relative_threshold
    return mzs[mask], ints[mask]

def anchor_peak_similarity(test_mz, test_mzs, test_ints, train_mzs_list, train_ints_list, train_precursors, mz_tolerance=0.01):
    test_mzs, test_ints = clean_spectrum(test_mzs, test_ints)
    if len(test_mzs) == 0:
        return np.zeros(len(train_mzs_list))
    
    # Identificar el Pico Base (el de mayor intensidad en el test)
    base_peak_idx = np.argmax(test_ints)
    test_base_mz = test_mzs[base_peak_idx]
    
    t_ints_sqrt = np.sqrt(test_ints)
    t_norm = np.linalg.norm(t_ints_sqrt)
    if t_norm == 0:
        return np.zeros(len(train_mzs_list))
    
    test_losses = test_mz - test_mzs
    scores = []
    
    for t_mz, t_ints, tr_prec in zip(train_mzs_list, train_ints_list, train_precursors):
        t_mz, t_ints = clean_spectrum(t_mz, t_ints)
        if len(t_mz) == 0:
            scores.append(0.0)
            continue
            
        tr_ints_sqrt = np.sqrt(t_ints)
        t_norm_tr = np.linalg.norm(tr_ints_sqrt)
        if t_norm_tr == 0:
            scores.append(0.0)
            continue
            
        # 1. Coseno de fragmentos
        diffs = np.abs(test_mzs[:, None] - t_mz[None, :])
        matches = diffs < mz_tolerance
        
        cos_score = 0.0
        if np.any(matches):
            dot_product = np.sum(np.max(matches * (t_ints_sqrt[:, None] * tr_ints_sqrt[None, :]), axis=1))
            cos_score = dot_product / (t_norm * t_norm_tr)
            
        # 2. Comprobación del Ancla (Pico Base coincidente)
        train_base_mz = t_mz[np.argmax(t_ints)]
        anchor_match = abs(test_base_mz - train_base_mz) < 0.008
        anchor_bonus = 0.25 if anchor_match else 0.0
            
        # 3. Pérdidas Neutras
        train_losses = tr_prec - t_mz
        loss_diffs = np.abs(test_losses[:, None] - train_losses[None, :])
        loss_matches = loss_diffs < mz_tolerance
        
        loss_score = 0.0
        if np.any(loss_matches):
            loss_dot = np.sum(np.max(loss_matches * (t_ints_sqrt[:, None] * tr_ints_sqrt[None, :]), axis=1))
            loss_score = loss_dot / (t_norm * t_norm_tr)
            
        # Puntuación combinada con el bonus crítico del Pico Base
        combined = (0.50 * cos_score + 0.25 * loss_score + anchor_bonus)
        scores.append(float(min(1.0, combined)))
            
    return np.array(scores)

print("3. Generando predicciones con Anclaje de Pico Base (500 candidatos)...")
predictions = {}

for _, row in test_df.iterrows():
    mol_id = row['molecule_id']
    test_mz = row['precursor_mz']
    test_adduct = row['adduct']
    
    test_ms2_mzs = row.get('ms2_mzs')
    test_ms2_ints = row.get('ms2_normalized_intensities')
    
    sub_db = db_by_adduct.get(test_adduct, train_df)
    if len(sub_db) == 0:
        sub_db = train_df
        
    diffs = np.abs(sub_db['precursor_mz'].values - test_mz)
    top_k_val = min(500, len(sub_db))
    top_idx = np.argsort(diffs)[:top_k_val]
    
    candidates_db = sub_db.iloc[top_idx].copy()
    cand_diffs = diffs[top_idx]
    
    if test_ms2_mzs is not None and len(test_ms2_mzs) > 0 and 'ms2_mzs' in candidates_db.columns:
        ms2_scores = anchor_peak_similarity(
            test_mz,
            test_ms2_mzs, test_ms2_ints,
            candidates_db['ms2_mzs'].values,
            candidates_db['ms2_normalized_intensities'].values,
            candidates_db['precursor_mz'].values,
            mz_tolerance=0.01
        )
    else:
        ms2_scores = np.zeros(len(candidates_db))

    mz_scores = np.exp(- (cand_diffs / 0.0022) ** 2)
    
    # 75% espectral con ancla, 25% masa para asegurar el Top 1
    final_scores = 0.75 * ms2_scores + 0.25 * mz_scores
    candidates_db['final_score'] = final_scores
    
    ranked = candidates_db.sort_values(by='final_score', ascending=False)
    best_candidates = ranked['normalized_smiles'].dropna().drop_duplicates().head(25).tolist()
    
    while len(best_candidates) < 25:
        best_candidates.append("CC(=O)O")
        
    predictions[mol_id] = ";".join(best_candidates[:25])

sub_df['smiles'] = sub_df['molecule_id'].map(predictions).fillna(fallback_smiles)
sub_df['smiles'] = sub_df['smiles'].apply(lambda x: fallback_smiles if pd.isna(x) or str(x).strip() == "" else x)

sub_df.to_csv("submission.csv", index=False)
print("--- SCRIPT DE ANCLAJE DE PICO BASE EJECUTADO CON ÉXITO ---")



1. Cargando datos para el asalto con Anclaje de Pico Base...
3. Generando predicciones con Anclaje de Pico Base (500 candidatos)...
--- SCRIPT DE ANCLAJE DE PICO BASE EJECUTADO CON ÉXITO ---
